In [1]:
%pip install -q mediapipe

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install mediapipe opencv-python

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#import libraries
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import time

In [ ]:
# ================== MODEL PATHS ==================
POSE_MODEL_PATH = "pose_landmarker_full.task"
HAND_MODEL_PATH = "hand_landmarker.task"

In [ ]:
# ================== BASE OPTIONS ==================
BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode

In [ ]:
# ================== POSE SETUP ==================
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions

pose_options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=POSE_MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1
)

In [ ]:
# ================== HAND SETUP ==================
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions

hand_options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=HAND_MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2
)

In [ ]:
# ================== DRAW CONNECTION FUNCTION ==================
def draw_connections(frame, landmarks, connections, w, h, color):
    for start_idx, end_idx in connections:
        start = landmarks[start_idx]
        end = landmarks[end_idx]

        x1, y1 = int(start.x * w), int(start.y * h)
        x2, y2 = int(end.x * w), int(end.y * h)

        cv2.line(frame, (x1, y1), (x2, y2), color, 2)

# ================== HAND CONNECTIONS ==================
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20)
]

# ================== POSE CONNECTIONS ==================
POSE_CONNECTIONS = [
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
    (11, 23), (12, 24)
]

In [ ]:



# ================== CAMERA ==================
cap = cv2.VideoCapture(0)
start_time = time.time()

window_name = "Pose + Hand Tracking"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

with PoseLandmarker.create_from_options(pose_options) as pose_landmarker, \
     HandLandmarker.create_from_options(hand_options) as hand_landmarker:

    while True:

        # Stop if window is closed
        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        timestamp_ms = int((time.time() - start_time) * 1000)

        # ================== POSE DETECTION ==================
        pose_result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)

        if pose_result.pose_landmarks:
            landmarks = pose_result.pose_landmarks[0]

            for idx in [11,12,13,14,15,16,23,24]:
                lm = landmarks[idx]
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 6, (0,255,0), -1)

            draw_connections(frame, landmarks, POSE_CONNECTIONS, w, h, (0,255,0))

        # ================== HAND DETECTION ==================
        hand_result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)

        open_percentage = 0
        closed_percentage = 0

        if hand_result.hand_landmarks:
            for hand_landmarks in hand_result.hand_landmarks:

                for lm in hand_landmarks:
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    cv2.circle(frame, (cx, cy), 5, (255,0,0), -1)

                draw_connections(frame, hand_landmarks, HAND_CONNECTIONS, w, h, (255,0,0))

                fingers_extended = 0

                # Index
                if hand_landmarks[8].y < hand_landmarks[5].y:
                    fingers_extended += 1

                # Middle
                if hand_landmarks[12].y < hand_landmarks[9].y:
                    fingers_extended += 1

                # Ring
                if hand_landmarks[16].y < hand_landmarks[13].y:
                    fingers_extended += 1

                # Pinky
                if hand_landmarks[20].y < hand_landmarks[17].y:
                    fingers_extended += 1

                open_percentage = int((fingers_extended / 4) * 100)
                closed_percentage = 100 - open_percentage

                cv2.putText(frame, f"Open: {open_percentage}%",
                            (30, 120),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.9, (0,255,0), 2)

                cv2.putText(frame, f"Closed: {closed_percentage}%",
                            (30, 160),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.9, (0,0,255), 2)

        # ================== DISPLAY ==================
        cv2.imshow(window_name, frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

# ================== CLEANUP ==================
cap.release()
cv2.destroyAllWindows()